In [ ]:
import json
import pathlib
import cv2
import warnings
import matplotlib

import spatial_scope
#import etl_bgt
import dataset
import evaluation

warnings.filterwarnings('ignore')
%matplotlib inline
matplotlib.rcParams.update({'font.size': 16})

In [ ]:
# Load configuration settings 
with open('satellite_imagery_config.json', 'r') as fp:
    config = json.load(fp)

In [ ]:
#with open('config.json', 'w') as fp:
#    json.dump(config, fp)

# Scope

In [ ]:
rtg = spatial_scope.NlRegionToGeom(**config['spatial_scope']['source'])
geom = rtg.from_region_name(config['spatial_scope']['region_name'])

In [ ]:
tbbox = spatial_scope.TiffBasedTiledBbox(geom,
                                         rtg.crs,
                                         config['satellite_image']['storage_dir'],
                                         config['satellite_image']['file_name'],
                                         config['satellite_image']['tile_size'])
#tbbox.write_geojson_of_bbox(config['satellite_image']['storage_dir'],
#                            config['satellite_image']['bbox_file'])
#tbbox.explore()

# Load transformed BGT data

In [ ]:
gdf = etl_bgt.process(geo_filter=tbbox.get_wkt_bbox_with_buffer(), crs=tbbox.crs)

In [ ]:
# Preview
gdf.clip(tbbox.id_to_bbox(8001)).explore(column='label') #tiles='CartoDB positron'

# Dataset creator

In [ ]:
dsc = dataset.DataSetCreator(tbbox, gdf,
                             in_storage_dir=config['satellite_image']['storage_dir'],
                             in_file_name=config['satellite_image']['file_name'],
                             out_storage_dir=config['dataset']['storage_dir'],
                             out_file_name=config['dataset']['agg_tile_data_file_name'],
                             featuretypes=config['bgt']['api_params']['featuretypes'])

In [ ]:
df = dsc.create()

In [ ]:
dsc.preview_tile(tile_id=2080)

# Exploratory data analysis

In [ ]:
eda = dataset.ExploratoryDataAnalysis(storage_dir=config['dataset']['storage_dir'],
                                      in_file_name=config['dataset']['agg_tile_data_file_name'],
                                      tbbox=tbbox,
                                      sat_image=dsc.data_array,
                                      out_file_name=config['dataset']['usable_tiles_data_file'])

In [ ]:
eda.satellite_and_tile_aggregate_side_by_side(column_name='contrast')

In [ ]:
eda.satellite_and_tile_aggregate_side_by_side(column_name='entropy')

In [ ]:
eda.four_mask_options()

In [ ]:
eda.set_mask(contrast_thld=0.98, entropy_thld=None)
#eda.masked_satellite_image()

In [ ]:
eda.entropy_histogram(write_to_file=True)

In [ ]:
eda.average_label_share(write_to_file=True)

In [ ]:
eda.pixel_count_bar(write_to_file=True)

In [ ]:
eda.export_unmasked_tiles()

# Data set splitting

In [ ]:
dss = dataset.DataSetSplitter(fractions=config['datasplit']['fractions'],
                              stratification_columns=config['datasplit']['stratification_columns'],
                              bins=config['datasplit']['bins'],
                              storage_dir=config['dataset']['storage_dir'],
                              agg_tile_data_file_name=config['dataset']['agg_tile_data_file_name'],
                              usable_tiles_file_name=config['dataset']['usable_tiles_data_file'],
                              data_split_file_name=config['datasplit']['data_split_file_name'])

In [ ]:
dss.find_best_split(range(100), write_to_file=True)

In [ ]:
dss.plot_distribution(write_to_file=True)

In [ ]:
dss.print_split_table()

In [ ]:
dss.write_data_split_to_file()

# Evaluation

In [ ]:
log_files = ['20221130125200_phase_1_logfile.json',
            '20221130135512_phase_2_logfile.json',
            '20221130145847_phase_3_logfile.json']
per_eval = evaluation.PerformanceEvaluation(config,
                                            log_files=log_files,
                                            tbbox=tbbox,
                                            geom=geom)

## Algemeen

In [ ]:
per_eval.plot_training_process()

In [ ]:
per_eval.pretty_print_confusion_matrices()

In [ ]:
per_eval.pretty_print_input_output_correlations(split='test')

## Geografisch: Gemeentegrens

In [ ]:
_ = per_eval.split_tiles_by_region(show_plot=True, write_to_file=False)

In [ ]:
per_eval.pretty_print_confusion_matrices_by_region()

## Geografisch: Wijken binnen de gemeentegrens

In [ ]:
per_eval.plot_performance_by_neighbourhood(metric='recall_onverhard', write_to_file=False)

In [ ]:
per_eval.plot_performance_by_neighbourhood(metric='recall_verhard', write_to_file=False)

In [ ]:
nbh_cm = per_eval.get_confusion_matrices_per_neighbourhood(dim=1)
nbh_cm.keys()

In [ ]:
per_eval.pretty_print_confusion_matrices_by_neighbourhood('Binnenstad', 'Zuid')

## Per tegel: Illustratieve voorbeelden

In [ ]:
# Get metadata of tile for search purposes
gdf = per_eval.join_all()
gdf.info()

### Binnenstad

In [ ]:
# Search for tile ids based on filters on metadata
mask = (gdf.split == 'test') & (gdf.nbh == 'Binnenstad') & (gdf.cm_1_0 > 10_000)
gdf[mask]

In [ ]:
per_eval.plot_input_labels_and_predicted_labels(tile_id=5837)

In [ ]:
per_eval.plot_input_labels_and_predicted_labels(tile_id=6194)

In [ ]:
per_eval.plot_input_labels_and_predicted_labels(tile_id=6672)

In [ ]:
per_eval.plot_input_labels_and_predicted_labels(tile_id=6798)

### Voorspellingen op onbekend terrein

In [ ]:
# Search for tile ids based on filters on metadata
mask = (gdf.split == 'test') & (gdf.onbekend > 50_000)
gdf[mask].head()

In [ ]:
per_eval.plot_input_labels_and_predicted_labels(tile_id=322)

In [ ]:
per_eval.plot_input_labels_and_predicted_labels(tile_id=472)